<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter5/Semantic_Search_with_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic search with FAISS (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install faiss-gpu

In [2]:
from datasets import load_dataset

issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 3019
})

In [3]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 808
})

In [4]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 808
})

In [5]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]

In [6]:
df["comments"][0].tolist()

['Cool, I think we can do both :)',
 '@lhoestq now the 2 are implemented.\r\n\r\nPlease note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).']

In [7]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"Cool, I think we can do both :)",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Protect master branch,@lhoestq now the 2 are implemented.\r\n\r\nPle...,After accidental merge commit (91c55355b634d0d...
2,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Hi ! I guess the caching mechanism should have...,## Describe the bug\r\nAfter upgrading to data...
3,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"If it's easy enough to implement, then yes ple...",## Describe the bug\r\nAfter upgrading to data...


In [8]:
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2964
})

In [9]:
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

In [10]:
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Filter:   0%|          | 0/2964 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 2175
})

In [11]:
def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }


comments_dataset = comments_dataset.map(concatenate_text)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
import torch

#device = torch.device("cuda")
device = torch.device("cpu")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
       

Now, we apply CLS pooling on the model's outputs, where we simply collect the last hidden state for the special [CLS] token. Thus, we create the following function:

In [14]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

Next, we'll create a helper function that will tokenize a list of documents, place the tensors on the GPU/CPU, feed them to the model, and finally apply CLS pooling to the outputs:

In [15]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

Next, we test that the function works by feeding it the first text entry in our corpus and inspecting the output shape:

In [16]:
embedding = get_embeddings(comments_dataset['text'][0])
embedding.shape

torch.Size([1, 768])

Great, now we converted the first entry in our corpus into a 768-dimensional vector! We can use the Dataset.map() to apply the get_embeddings() function to each row in the corpus, so let's create a new embeddings column as follows:

In [17]:
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(['text']).detach().cpu().numpy()[0]} # In PyTorch, the sequence .detach().cpu().numpy() is the standard pipeline used to safely convert a GPU-based tensor into a standard NumPy array.
)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [23]:
embeddings_dataset, embeddings_dataset['embeddings']

(Dataset({
     features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
     num_rows: 2175
 }),
 Column([[0.07118327915668488, -0.5791645050048828, -0.36235636472702026, 0.2813660502433777, 0.03646588325500488, -0.10325802117586136, 0.23238718509674072, 0.06695636361837387, 0.03650382161140442, 0.30037620663642883, -0.06511244177818298, -0.17731942236423492, -0.007691849488765001, 0.13963381946086884, -0.23413242399692535, -0.10203982144594193, -0.05582743510603905, 0.1252112090587616, 0.05229572951793671, -0.010527738370001316, 0.06613259762525558, 0.0012292292667552829, -0.013745809905230999, 0.18881835043430328, -0.026697644963860512, -0.12208615243434906, -0.08225010335445404, -0.2036672830581665, -0.09114796668291092, -0.224282905459404, -0.2314019501209259, 0.16289442777633667, -0.3214252293109894, 0.10572753101587296, -9.030225919559598e-05, -0.4025554358959198, 0.06144044175744057, -0.10525675117969513, -0.05274441838264465, -0.231625378131

Now that we have a dataset of embeddings, we need some way to search over them. To do this we'll use a special data structure called a FAISS index.

FAISS (short for Facebook AI Similarity Search) is a library that provides efficient algorithms to quickly search and cluster embedding vectors.

The basic idea behind FAISS is to create a special data structure called an index that allows one to find which embeddings are similar to an input embedding.

Creating a FAISS index in Hugging Face Datasets is simple: we use the Dataset.add_faiss_index() function and specify which column of our dataset we'd like to index:

In [22]:
import sys
import torchvision.io

# Create a real, empty class to safely pass isinstance() checks
class DummyVideoReader:
    pass

torchvision.io.VideoReader = DummyVideoReader

import datasets

embeddings_dataset.add_faiss_index(column='embeddings')

  0%|          | 0/3 [00:00<?, ?it/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
    num_rows: 2175
})

Now we can perform queries on this index by doing a NN lookup with the Dataset.get_nearest_examples() function. Let's test this out by first embedding a question as follows:

In [35]:
question = "How can I load Datasets?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

Just like with the documents, we now have a 768-dimensional vector representing the query, which we can compare against the whole corpus to find the most similar embeddings

In [36]:
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

The Dataset.get_nearest_examples() returns a tuple of scores that rank the overlap between the query and the document, and a corresponding set of samples (here we have the 5 best matches, hence k=5).

Let's collect these in a pandas.DataFrame so we can easily sort them:

In [37]:
scores

array([52.17971, 52.17971, 52.17971, 52.17971, 52.17971], dtype=float32)

In [38]:
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df['scores'] = scores
samples_df.sort_values('scores', ascending=False, inplace=True)

In [39]:
for _, row in samples_df.iterrows():
  print(f"COMMENT: {row.comments}")
  print(f"COMMENT: {row.scores}")
  print(f"URL: {row.html_url}")
  print(f"=" * 50)
  print()

COMMENT: @lhoestq now the 2 are implemented.

Please note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).
COMMENT: 52.179710388183594
URL: https://github.com/huggingface/datasets/issues/2945

COMMENT: Hi ! I guess the caching mechanism should have considered the new `filter` to be different from the old one, and don't use cached results from the old `filter`.
To avoid other users from having this issue we could make the caching differentiate the two, what do you think ?
COMMENT: 52.179710388183594
URL: https://github.com/huggingface/datasets/issues/2943

COMMENT: If it's easy enough to implement, then yes please 😄  But this issue can be low-priority, since I've only encountered it 